# Bases de données — Exercice 2 : les aéroports (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue : la requête SQL à taper dans la **console H2**, et le résultat obtenu |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** `SELECT ... FROM ... WHERE`, `DISTINCT`, `COUNT`, `MAX`, `MIN`, `AND`, `LIKE`, jointures (`JOIN ... ON`), requêtes imbriquées, `ORDER BY`, et pour finir `GROUP BY`.

**Données :** l'archive `airports_sql.zip` du site du professeur, décompressée : le dossier `airports_sql/` est placé à côté de ce notebook.

**Question 5 :** le fichier `airports_tables_v2.sql` n'est pas encore disponible sur le site du professeur (lien cassé). La question 5 n'est donc pas traitée ; les questions de la partie 6 n'utilisent que des attributs déjà présents dans les tables, elles sont résolues avec les données actuelles.

> ⚠️ **Liberté prise :** dans la question 3, on s'autorise plusieurs requêtes successives (chercher un code, puis l'utiliser) ; la question 4 les regroupe en une seule requête, comme le demande le sujet.

---
# MISE EN PLACE

En classe, les requêtes SQL sont tapées dans la **console H2** (le SGBD utilisé dans le cours). C'est aussi le cas ici : **chaque requête de ce notebook a été exécutée dans H2**, et le résultat affiché sous la requête est celui qu'a donné H2. Il n'y a **aucun code à exécuter** dans ce notebook : il suffit de recopier les requêtes dans la console H2.

**Pour ouvrir la console H2 :** lancer « H2 Console », se connecter avec la JDBC URL `jdbc:h2:~/nsi`, l'utilisateur `sa` et un mot de passe vide.

> ⚠️ **Liberté prise :** les résultats ont été obtenus avec H2 version 2.5.252 et recopiés tels quels, à trois détails près : la durée d'exécution affichée par H2 (« 12 ms ») a été supprimée, car elle change à chaque fois ; pour les résultats longs, seules les 12 premières lignes sont recopiées, suivies de « ... » et du nombre total de lignes ; dans les messages d'erreur, le nom technique de l'erreur Java (`org.h2.jdbc....Exception`) a été remplacé par « Erreur : » ; quand plusieurs instructions sont exécutées d'un coup, chacune est rappelée au-dessus de son résultat. Dans la console du navigateur, le résultat apparaît sous forme de tableau ; le contenu est le même.

---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Télécharger l'archive airports_sql.zip : il contient les fichiers airports_tables.sql et airports_data.sql : exécuter ces fichiers dans cet ordre dans la console H2.

#### ✅ RÉSOLUTION

On décompresse l'archive, puis, dans la console H2 (en remplaçant `/chemin/vers/` par le dossier où se trouve `airports_sql`) :

```sql
RUNSCRIPT FROM '/chemin/vers/airports_sql/airports_tables.sql';
RUNSCRIPT FROM '/chemin/vers/airports_sql/airports_data.sql';
```

*Résultat dans la console H2 :*

```text
RUNSCRIPT FROM '/chemin/vers/airports_sql/airports_tables.sql';
(Update count: 9)

RUNSCRIPT FROM '/chemin/vers/airports_sql/airports_data.sql';
(Update count: 122021)
```

> ⚠️ **Liberté prise :** non demandé par le sujet, mais indispensable dans H2 : exécuter aussi ces quatre lignes, **une seule fois** après le chargement. Sans elles, les requêtes avec jointure (questions 3 (j), 4 et 6) peuvent prendre **plusieurs minutes** (voir l'explication ci-dessous). `CREATE INDEX` est hors programme.

```sql
CREATE INDEX ON Countries(code);
CREATE INDEX ON Airports(id);
CREATE INDEX ON Regions(code);
CREATE INDEX ON Runways(id);
```

*Résultat dans la console H2 :*

```text
CREATE INDEX ON Countries(code);
(Update count: 0)

CREATE INDEX ON Airports(id);
(Update count: 0)

CREATE INDEX ON Regions(code);
(Update count: 0)

CREATE INDEX ON Runways(id);
(Update count: 0)
```

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT COUNT(*) FROM Countries;
SELECT COUNT(*) FROM Regions;
SELECT COUNT(*) FROM Airports;
SELECT COUNT(*) FROM Runways;
```

*Résultat dans la console H2 :*

```text
SELECT COUNT(*) FROM Countries;
COUNT(*)
248
(1 row)

SELECT COUNT(*) FROM Regions;
COUNT(*)
3862
(1 row)

SELECT COUNT(*) FROM Airports;
COUNT(*)
73904
(1 row)

SELECT COUNT(*) FROM Runways;
COUNT(*)
44006
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Pourquoi cet ordre ?** `airports_tables.sql` contient les instructions `CREATE TABLE` : il décrit la **structure** de la base (le *schéma* : les relations et leurs attributs). `airports_data.sql` contient des milliers d'instructions `INSERT INTO` : c'est le **contenu**. On ne peut pas ranger des données dans une table qui n'existe pas encore : il faut d'abord créer les tables.
- C'est exactement la distinction du cours : **structure** d'une base de données (le schéma) ≠ **contenu** (les enregistrements).
- Le fichier `airports_tables.sql` commence par des `DROP TABLE IF EXISTS ...;` : si les tables existent déjà (par exemple après l'exercice 1), elles sont supprimées puis recréées. On peut donc relancer les deux fichiers sans créer de doublons.
- On obtient 248 pays, 3 862 régions, 73 904 aéroports et 44 006 pistes.
- **Pourquoi les index ?** Les tables de `airports_tables.sql` n'ont **pas de clé primaire**. Pour une jointure entre `Runways` (44 006 pistes) et `Airports` (73 904 aéroports), H2 compare alors chaque piste à chaque aéroport : plus de 3 milliards de comparaisons ! Un **index** est une sorte de table des matières : il permet au SGBD de trouver directement l'aéroport d'identifiant donné, sans tout parcourir. Avec les index, la même requête passe de plusieurs minutes à une fraction de seconde. C'est un des services rendus par un SGBD cités dans le cours : l'**efficacité de traitement des requêtes**. (Certains SGBD créent ces index tout seuls pendant une jointure ; H2 non.)
- Déclarer une **clé primaire** crée automatiquement un index : c'est sans doute l'un des buts du fichier `airports_tables_v2.sql` de la question 5. `CREATE INDEX` est hors programme.

---
# QUESTION 2

> La commande SELECT permet d'extraire des données des relations (tables) de la base de données.
> Donner la syntaxe des instructions qui permettent d'afficher les données ci-dessous.

## 2 (a)

#### 📝 ÉNONCÉ

> (a) La relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT * FROM Countries;
```

*Résultat dans la console H2 :*

```text
ID     | CODE | NAME                                         | CONTINENT
302672 | AD   | Andorra                                      | EU
302618 | AE   | United Arab Emirates                         | AS
302619 | AF   | Afghanistan                                  | AS
302722 | AG   | Antigua and Barbuda                          | NA
302723 | AI   | Anguilla                                     | NA
302673 | AL   | Albania                                      | EU
302620 | AM   | Armenia                                      | AS
302556 | AO   | Angola                                       | AF
302615 | AQ   | Antarctica                                   | AN
302789 | AR   | Argentina                                    | SA
302763 | AS   | American Samoa                               | OC
302674 | AT   | Austria                                      | EU
...
(248 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- L'étoile `*` signifie « **tous les attributs** ». Sans `WHERE`, on obtient **tous les enregistrements** : les 248 pays (seules les premières lignes sont recopiées ci-dessus).
- Chaque ligne est un **enregistrement** (un pays), chaque colonne un **attribut** (`id`, `code`, `name`, `continent`).

## 2 (b)

#### 📝 ÉNONCÉ

> (b) Les noms de tous les pays enregistrés dans la relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Countries;
```

*Résultat dans la console H2 :*

```text
NAME
Andorra
United Arab Emirates
Afghanistan
Antigua and Barbuda
Anguilla
Albania
Armenia
Angola
Antarctica
Argentina
American Samoa
Austria
...
(248 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Après `SELECT`, on écrit **les attributs que l'on veut voir** : ici seulement `name`. Le résultat a toujours 248 lignes, mais une seule colonne.
- ⚠️ Les noms des attributs sont ceux du schéma, **en anglais** (`name`, pas `nom`).

## 2 (c)

#### 📝 ÉNONCÉ

> (c) Les noms et les codes ISO de tous les pays renseignés dans la relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT name, code FROM Countries;
```

*Résultat dans la console H2 :*

```text
NAME                                         | CODE
Andorra                                      | AD
United Arab Emirates                         | AE
Afghanistan                                  | AF
Antigua and Barbuda                          | AG
Anguilla                                     | AI
Albania                                      | AL
Armenia                                      | AM
Angola                                       | AO
Antarctica                                   | AQ
Argentina                                    | AR
American Samoa                               | AS
Austria                                      | AT
...
(248 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Plusieurs attributs sont **séparés par des virgules**. Les colonnes du résultat apparaissent **dans l'ordre où on les écrit** (ici le nom, puis le code), pas dans l'ordre de la table.

## 2 (d)

#### 📝 ÉNONCÉ

> (d) Les codes des continents renseignés dans la relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT DISTINCT continent FROM Countries;
```

*Résultat dans la console H2 :*

```text
CONTINENT
AF
AN
AS
EU
NA
OC
SA
(7 rows)
```

**Réponse :** 7 codes : `EU`, `AS`, `NA`, `AF`, `AN`, `SA`, `OC`.

#### 💡 EXPLICATION PÉDAGOGIQUE

- Sans `DISTINCT`, on obtiendrait **248 lignes** (une par pays) : `EU` apparaîtrait des dizaines de fois.
- Le mot-clé `DISTINCT` **supprime les doublons** du résultat : chaque code n'apparaît qu'une fois.
- Il se place **juste après `SELECT`**.

## 2 (e)

#### 📝 ÉNONCÉ

> (e) L'identifiant de la France dans la relation Countries.

#### ✅ RÉSOLUTION

```sql
SELECT id FROM Countries WHERE name = 'France';
```

*Résultat dans la console H2 :*

```text
ID
302687
(1 row)
```

**Réponse :** 302687.

#### 💡 EXPLICATION PÉDAGOGIQUE

- La clause `WHERE` **filtre** les enregistrements : on ne garde que ceux qui vérifient la condition.
- En SQL, le test d'égalité s'écrit avec **un seul** `=` (et non `==` comme en Python).
- Les chaînes de caractères sont entre **guillemets simples** `'France'`. ⚠️ Pas d'espace à l'intérieur : `' France '` serait une autre chaîne (avec des espaces) et la requête ne trouverait rien. C'est le piège du copier-coller depuis le PDF.
- On aurait aussi pu écrire `WHERE code = 'FR'`.

## 2 (f)

#### 📝 ÉNONCÉ

> (f) Les coordonnées GPS de l'aéroport d'Orly.

#### ✅ RÉSOLUTION

On commence par **chercher** l'aéroport, car on ne connaît pas son nom exact ni son code :

```sql
SELECT code, name, city FROM Airports WHERE name LIKE '%Orly%';
```

Trois aéroports contiennent « Orly » : celui de Paris a le code `LFPO`. D'où la requête :

```sql
SELECT latitude, longitude FROM Airports WHERE code = 'LFPO';
```

*Résultat dans la console H2 :*

```text
LATITUDE | LONGITUDE
48.72333 | 2.3794444
(1 row)
```

**Réponse :** latitude 48.7233333, longitude 2.3794444.

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT code, name, city FROM Airports WHERE name LIKE '%Orly%';
```

*Résultat dans la console H2 :*

```text
CODE    | NAME                       | CITY
KE-0034 | Orly (Olooitikosh) Airport | Orly (Olooitikosh)
LFPO    | Paris-Orly Airport         | Paris
(2 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Les coordonnées GPS sont deux attributs de la relation `Airports` : `latitude` et `longitude`.
- **`LIKE` avec `%`** permet de chercher un *morceau* de chaîne : `'%Orly%'` signifie « n'importe quoi, puis Orly, puis n'importe quoi ». C'est très pratique pour explorer les données.
- ⚠️ **Piège :** `WHERE name LIKE '%Orly%'` renvoie **3 aéroports**, dont un au Kenya (*Orly (Olooitikosh) Airport*) et une base fermée en Russie. Il faut ensuite filtrer sur une valeur **unique** : le code ICAO `'LFPO'` (ou le nom exact `'Paris-Orly Airport'`).
- ⚠️ Dans H2, `LIKE` **distingue majuscules et minuscules** : `'%orly%'` ne trouverait rien.

## 2 (g)

#### 📝 ÉNONCÉ

> (g) Le nombre de régions renseignées dans la relation Regions.

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Regions;
```

*Résultat dans la console H2 :*

```text
COUNT(*)
3862
(1 row)
```

**Réponse :** 3 862 régions.

#### 💡 EXPLICATION PÉDAGOGIQUE

- `COUNT(*)` est une **fonction d'agrégation** : au lieu d'afficher les enregistrements, elle les **compte**. Le résultat est une seule valeur.
- `COUNT(*)` compte les lignes ; `COUNT(name)` compterait les lignes où `name` est renseigné (non `NULL`). Ici, c'est pareil.

## 2 (h)

#### 📝 ÉNONCÉ

> (h) La plus grande longueur de piste renseignée dans la relation Runways.

#### ✅ RÉSOLUTION

```sql
SELECT MAX(length) FROM Runways;
```

*Résultat dans la console H2 :*

```text
MAX(LENGTH)
30000
(1 row)
```

**Réponse :** 30 000.

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT * FROM Runways WHERE length = 30000;
```

*Résultat dans la console H2 :*

```text
ID     | AIRPORT_ID | TYPE  | LENGTH | WIDTH | LIGHTED | CLOSED
256946 | 22159      | WATER | 30000  | 5000  | 0       | 0
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- `MAX` est une autre fonction d'agrégation (comme `MIN`, `SUM`, `AVG`, `COUNT`).
- **Unité :** le jeu de données *OurAirports* donne les longueurs et largeurs **en pieds** (1 pied ≈ 0,3048 m). La plus longue piste de Charles de Gaulle mesure 13 829 pieds, soit 4 215 m.
- **Regard critique :** la piste de 30 000 pieds (plus de 9 km !) est une piste **sur l'eau** (`type = 'WATER'`) d'une base d'hydravions aux États-Unis : ce n'est pas une vraie piste en dur. Une donnée peut être juste pour le SGBD et pourtant surprenante : il faut toujours **regarder** ses résultats.

---
# QUESTION 3

> En utilisant le jeu de données fourni, répondre aux questions suivantes à l'aide de requêtes SQL.

Dans cette question, on s'autorise **plusieurs requêtes successives** quand c'est naturel (on cherche d'abord un code ou un identifiant, puis on l'utilise). La question 4 demande ensuite de tout faire en **une seule** requête.

## 3 (a)

#### 📝 ÉNONCÉ

> (a) Quel est le pays dont le code ISO est IM ?

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Countries WHERE code = 'IM';
```

*Résultat dans la console H2 :*

```text
NAME
Isle of Man
(1 row)
```

**Réponse :** *Isle of Man* (l'île de Man).

#### 💡 EXPLICATION PÉDAGOGIQUE

- Requête de base : on connaît la valeur d'un attribut (`code`), on veut la valeur d'un autre attribut (`name`) **du même enregistrement**.

## 3 (b)

#### 📝 ÉNONCÉ

> (b) Quel est le code ICAO de l'aéroport de Toulouse-Blagnac ?

#### ✅ RÉSOLUTION

On cherche d'abord le nom exact (les noms sont en anglais) :

```sql
SELECT code, name FROM Airports WHERE name LIKE '%Blagnac%';
```

puis :

```sql
SELECT code FROM Airports WHERE name = 'Toulouse-Blagnac Airport';
```

*Résultat dans la console H2 :*

```text
CODE
LFBO
(1 row)
```

**Réponse :** `LFBO`.

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT code, name FROM Airports WHERE name LIKE '%Blagnac%';
```

*Résultat dans la console H2 :*

```text
CODE | NAME
LFBO | Toulouse-Blagnac Airport
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Dans la relation `Airports`, l'attribut `code` est le code **ICAO** de l'aéroport (4 lettres ; ceux de France commencent par `LF`).
- `WHERE name LIKE '%Toulouse%'` renverrait **5 aéroports** (Toulouse-Francazal, Toulouse-Lasbordes...). Chercher `'%Blagnac%'` est plus précis.
- La première requête suffit d'ailleurs à répondre ; la seconde montre la requête « propre » avec le nom exact.

## 3 (c)

#### 📝 ÉNONCÉ

> (c) Quels sont les noms des héliports de la capitale de l'Allemagne ?

#### ✅ RÉSOLUTION

Le code ISO de l'Allemagne s'obtient avec `SELECT code FROM Countries WHERE name = 'Germany';` → `DE`. Puis :

```sql
SELECT name FROM Airports WHERE type = 'heliport' AND city = 'Berlin' AND country = 'DE';
```

*Résultat dans la console H2 :*

```text
NAME
Berlin Emergency Hospital Heliport
Helios Clinic Berlin-Buch Heliport
Vivantes Humboldt Clinic Heliport
Charité Clinic Virchow Campus Heliport
Vivantes Freidrichshain Clinic Heliport
Berlin Tempelhof Heliport
Charité Clinic Benjamin Franklin Campus Heliport
Havelhöhe Medical Heliport
(8 rows)
```

**Réponse :** 8 héliports (Berlin Emergency Hospital Heliport, Charité Clinic Virchow Campus Heliport, Berlin Tempelhof Heliport, ...).

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT name, country FROM Airports WHERE type = 'heliport' AND city = 'Berlin';
```

*Résultat dans la console H2 :*

```text
NAME                                             | COUNTRY
Berlin Fairgrounds Heliport                      | US
AGH Heliport                                     | US
Androscoggin Valley Hospital Helipad             | US
Berlin Emergency Hospital Heliport               | DE
Helios Clinic Berlin-Buch Heliport               | DE
Vivantes Humboldt Clinic Heliport                | DE
Charité Clinic Virchow Campus Heliport           | DE
Vivantes Freidrichshain Clinic Heliport          | DE
Berlin Tempelhof Heliport                        | DE
Charité Clinic Benjamin Franklin Campus Heliport | DE
Havelhöhe Medical Heliport                       | DE
West Jersey Hospital Heliport                    | US
(12 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- On combine plusieurs conditions avec **`AND`** : un enregistrement est gardé seulement si **toutes** les conditions sont vraies.
- Les types d'aéroports sont écrits en anglais et en minuscules : `'heliport'`, `'small_airport'`, etc. (voir `SELECT DISTINCT type FROM Airports;`).
- ⚠️ **Le piège :** sans `country = 'DE'`, on obtient **12** héliports, car il existe aussi des villes nommées *Berlin* aux **États-Unis** (New Hampshire, New Jersey...). Une ville n'est pas identifiée par son seul nom : il faut préciser le pays.

## 3 (d)

#### 📝 ÉNONCÉ

> (d) Combien de régions sont répertoriées au Brésil ?

#### ✅ RÉSOLUTION

Code du Brésil : `SELECT code FROM Countries WHERE name = 'Brazil';` → `BR`. Puis :

```sql
SELECT COUNT(*) FROM Regions WHERE country = 'BR';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
28
(1 row)
```

**Réponse :** 28 régions (dont une région « (unassigned) », donc 27 vraies régions : les 26 États et le District fédéral).

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT code, name FROM Regions WHERE country = 'BR';
```

*Résultat dans la console H2 :*

```text
CODE   | NAME
BR-AC  | Acre
BR-AL  | Alagoas
BR-AM  | Amazonas
BR-AP  | Amapá
BR-BA  | Bahia
BR-CE  | Ceará
BR-DF  | Distrito Federal
BR-ES  | Espírito Santo
BR-GO  | Goiás
BR-MA  | Maranhão
BR-MG  | Minas Gerais
BR-MS  | Mato Grosso do Sul
...
(28 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- `COUNT(*)` et `WHERE` se combinent : le SGBD **filtre** d'abord (les régions du Brésil), **puis compte**.
- Dans la relation `Regions`, l'attribut `country` contient le **code ISO** du pays (`'BR'`), pas son nom : `WHERE country = 'Brazil'` donnerait 0.

## 3 (e)

#### 📝 ÉNONCÉ

> (e) Combien d'aéroports ont une altitude négative ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Airports WHERE altitude < 0;
```

*Résultat dans la console H2 :*

```text
COUNT(*)
90
(1 row)
```

**Réponse :** 90 aéroports.

#### 💡 EXPLICATION PÉDAGOGIQUE

- Les comparaisons `<`, `>`, `<=`, `>=` s'utilisent comme en Python ; « différent de » s'écrit `<>` en SQL.
- Une altitude négative signifie que l'aéroport est **sous le niveau de la mer** (par exemple Amsterdam-Schiphol, ou autour de la mer Morte).

## 3 (f)

#### 📝 ÉNONCÉ

> (f) Combien la France compte-t-elle d'héliports ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Airports WHERE country = 'FR' AND type = 'heliport';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
265
(1 row)
```

**Réponse :** 265 héliports.

#### 💡 EXPLICATION PÉDAGOGIQUE

- Même schéma qu'en (c) et (d) : un filtre avec `AND`, puis un comptage.

## 3 (g)

#### 📝 ÉNONCÉ

> (g) Quelles sont les longueurs des pistes de l'aéroport Charles de Gaulle ?

#### ✅ RÉSOLUTION

Les pistes sont dans la relation `Runways`, mais l'aéroport y est désigné par son **identifiant** `airport_id`. On cherche d'abord cet identifiant (code ICAO trouvé avec `name LIKE '%Gaulle%'`), puis les pistes :

```sql
SELECT id FROM Airports WHERE code = 'LFPG';
SELECT length FROM Runways WHERE airport_id = 4185;
```

*Résultat dans la console H2 :*

```text
SELECT id FROM Airports WHERE code = 'LFPG';
ID
4185
(1 row)

SELECT length FROM Runways WHERE airport_id = 4185;
LENGTH
13829
8858
8858
13780
(4 rows)
```

**Réponse :** 4 pistes : 13 829, 8 858, 8 858 et 13 780 pieds (soit environ 4 215 m, 2 700 m, 2 700 m et 4 200 m).

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT id, code, name FROM Airports WHERE name LIKE '%Gaulle%';
```

*Résultat dans la console H2 :*

```text
ID   | CODE | NAME
4185 | LFPG | Charles de Gaulle International Airport
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Les informations sont réparties dans **deux relations** : l'aéroport dans `Airports`, ses pistes dans `Runways`. Le lien entre les deux est l'attribut `airport_id` de `Runways`, qui contient l'`id` d'un aéroport : c'est une **clé étrangère**.
- Ici, on fait le lien « à la main » en deux requêtes. La question 4 montre comment le faire en une seule requête avec `JOIN`.
- Deux pistes ont la même longueur (8 858) : on les voit toutes les deux, car `SELECT` sans `DISTINCT` garde les doublons, ce qui est voulu ici (ce sont deux pistes différentes).

## 3 (h)

#### 📝 ÉNONCÉ

> (h) Dans quel pays se trouve l'aéroport le plus au sud ?

#### ✅ RÉSOLUTION

Le plus au sud, c'est la **plus petite latitude**. En trois étapes :

```sql
SELECT MIN(latitude) FROM Airports;
SELECT name, country FROM Airports WHERE latitude = -90;
SELECT name FROM Countries WHERE code = 'AQ';
```

*Résultat dans la console H2 :*

```text
SELECT MIN(latitude) FROM Airports;
MIN(LATITUDE)
-90.0
(1 row)

SELECT name, country FROM Airports WHERE latitude = -90;
NAME                       | COUNTRY
South Pole Station Airport | AQ
(1 row)

SELECT name FROM Countries WHERE code = 'AQ';
NAME
Antarctica
(1 row)
```

**Réponse :** *Antarctica* (l'Antarctique) : c'est l'aéroport de la station Amundsen-Scott, exactement au pôle Sud (latitude −90).

#### 💡 EXPLICATION PÉDAGOGIQUE

- La latitude va de −90 (pôle Sud) à +90 (pôle Nord) : « le plus au sud » = `MIN(latitude)`.
- ⚠️ On ne peut **pas** écrire `SELECT country, MIN(latitude) FROM Airports;` en espérant obtenir le pays de l'aéroport le plus au sud : mélanger un attribut et une fonction d'agrégation sans regroupement est une erreur dans H2 (et donne un résultat non garanti dans d'autres SGBD).
- Le pays obtenu dans `Airports` est un **code** (`AQ`) : il faut une troisième requête (ou une jointure) pour obtenir son nom.

## 3 (i)

#### 📝 ÉNONCÉ

> (i) Quelle est la ville où se trouve la piste en asphalte la plus longue ?

#### ✅ RÉSOLUTION

```sql
SELECT MAX(length) FROM Runways WHERE type = 'ASP';
SELECT airport_id FROM Runways WHERE type = 'ASP' AND length = 16076;
SELECT city FROM Airports WHERE id = 2847;
```

*Résultat dans la console H2 :*

```text
SELECT MAX(length) FROM Runways WHERE type = 'ASP';
MAX(LENGTH)
16076
(1 row)

SELECT airport_id FROM Runways WHERE type = 'ASP' AND length = 16076;
AIRPORT_ID
2847
(1 row)

SELECT city FROM Airports WHERE id = 2847;
CITY
Upington
(1 row)
```

**Réponse :** *Upington* (Afrique du Sud) : piste de 16 076 pieds (environ 4,9 km).

> ⚠️ **Liberté prise :** on s'en tient au code `'ASP'` donné par le sujet, sans les autres écritures de l'asphalte présentes dans les données (`'ASPH'`, `'Asphalt'`...). Même choix pour toutes les questions sur l'asphalte.

#### 💡 EXPLICATION PÉDAGOGIQUE

- Le sujet indique « asp : asphalte », mais dans les données les codes sont **en majuscules** : `'ASP'`. Avec `type = 'asp'`, on ne trouverait rien. On le découvre avec `SELECT DISTINCT type FROM Runways;`.
- **Regard critique :** les données d'*OurAirports* sont saisies par des contributeurs, et on trouve aussi `'ASPH'`, `'Asphalt'`, `'asphalt'`... pour la même chose. On s'en tient ici au code donné par le sujet.
- Même méthode qu'en (h) : on calcule d'abord le maximum, puis on cherche l'enregistrement qui a cette valeur.

## 3 (j)

#### 📝 ÉNONCÉ

> (j) Quelle est la ville d'Europe où se trouve la piste en asphalte la plus longue ?

#### ✅ RÉSOLUTION

Le continent n'est pas dans `Runways`, ni dans `Airports` : il est dans `Countries`. Il faut donc relier les **trois** relations avec des jointures :
- `Runways.airport_id` fait référence à `Airports.id` ;
- `Airports.country` fait référence à `Countries.code`.

```sql
SELECT MAX(Runways.length) FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU';
SELECT Airports.city FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU' AND Runways.length = 13829;
```

*Résultat dans la console H2 :*

```text
SELECT MAX(Runways.length) FROM Runways JOIN Airports ON Airports.id = Runways.airport_id JOIN Countries ON Countries.code = Airports.country WHERE Runways.type = 'ASP' AND Countries.continent = 'EU';
MAX(RUNWAYS.LENGTH)
13829
(1 row)

SELECT Airports.city FROM Runways JOIN Airports ON Airports.id = Runways.airport_id JOIN Countries ON Countries.code = Airports.country WHERE Runways.type = 'ASP' AND Countries.continent = 'EU' AND Runways.length = 13829;
CITY
Paris
(1 row)
```

**Réponse :** *Paris* (aéroport Charles de Gaulle, piste de 13 829 pieds, soit 4 215 m).

#### 💡 EXPLICATION PÉDAGOGIQUE

- **La jointure** `R1 JOIN R2 ON condition` met « côte à côte » les enregistrements de R1 et de R2 qui vérifient la condition. Ici, chaque piste est collée à son aéroport, puis au pays de cet aéroport : on peut alors filtrer sur des attributs des trois relations à la fois.
- Quand plusieurs relations ont un attribut du même nom (`id`, `code`, `name`, `type`, `country`...), on **préfixe** par le nom de la relation : `Runways.type` (type de piste) n'est pas `Airports.type` (type d'aéroport).
- ⚠️ La condition `Countries.continent = 'EU'` doit figurer dans les **deux** requêtes : sinon, une piste de 13 829 pieds hors d'Europe serait aussi trouvée.

---
# QUESTION 4

#### 📝 ÉNONCÉ

> 4. Si ce n'était pas le cas, répondre aux questions précédentes en utilisant une seule requête SQL.

Les questions (a), (b), (e) et (f) sont déjà résolues en une seule requête. Il reste (c), (d), (g), (h), (i) et (j). Deux outils :
- la **jointure** (`JOIN`), au programme, pour aller chercher une information dans une autre relation au lieu de recopier un code ou un identifiant à la main ;
- la **requête imbriquée** (une requête entre parenthèses à l'intérieur d'une autre), **hors programme** mais présentée page 8 du polycopié, pour utiliser un maximum ou un minimum sans le recopier.

## 4 — (c) héliports de Berlin

#### 📝 ÉNONCÉ

> (c) Quels sont les noms des héliports de la capitale de l'Allemagne ?

#### ✅ RÉSOLUTION

```sql
SELECT Airports.name FROM Airports
    JOIN Countries ON Countries.code = Airports.country
    WHERE Airports.type = 'heliport' AND Airports.city = 'Berlin' AND Countries.name = 'Germany';
```

*Résultat dans la console H2 :*

```text
NAME
Berlin Emergency Hospital Heliport
Helios Clinic Berlin-Buch Heliport
Vivantes Humboldt Clinic Heliport
Charité Clinic Virchow Campus Heliport
Vivantes Freidrichshain Clinic Heliport
Berlin Tempelhof Heliport
Charité Clinic Benjamin Franklin Campus Heliport
Havelhöhe Medical Heliport
(8 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Au lieu de chercher le code `DE` puis de le recopier, on **joint** `Airports` et `Countries` et on filtre directement sur le nom du pays.
- `Airports.name` (nom de l'aéroport) et `Countries.name` (nom du pays) : le préfixe est **obligatoire**, sinon le SGBD ne sait pas de quel `name` on parle (erreur « ambiguous column »).

## 4 — (d) régions du Brésil

#### 📝 ÉNONCÉ

> (d) Combien de régions sont répertoriées au Brésil ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Regions
    JOIN Countries ON Countries.code = Regions.country
    WHERE Countries.name = 'Brazil';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
28
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Même idée : `Regions.country` contient un code de pays qui fait référence à `Countries.code`.

## 4 — (g) pistes de Charles de Gaulle

#### 📝 ÉNONCÉ

> (g) Quelles sont les longueurs des pistes de l'aéroport Charles de Gaulle ?

#### ✅ RÉSOLUTION

```sql
SELECT Runways.length FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    WHERE Airports.code = 'LFPG';
```

*Résultat dans la console H2 :*

```text
LENGTH
13829
8858
8858
13780
(4 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- La condition de jointure `Airports.id = Runways.airport_id` relie chaque piste à **son** aéroport : c'est la clé étrangère `airport_id` qui fait référence à la clé `id`.
- ⚠️ **Erreur fréquente :** oublier le `ON ...`, ou écrire une mauvaise condition (`Airports.id = Runways.id`) : le SGBD associerait alors des pistes à des aéroports qui n'ont rien à voir.

## 4 — (h) aéroport le plus au sud

#### 📝 ÉNONCÉ

> (h) Dans quel pays se trouve l'aéroport le plus au sud ?

#### ✅ RÉSOLUTION

```sql
SELECT Countries.name FROM Countries
    JOIN Airports ON Countries.code = Airports.country
    WHERE Airports.latitude = (SELECT MIN(latitude) FROM Airports);
```

*Résultat dans la console H2 :*

```text
NAME
Antarctica
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Requête imbriquée :** le SGBD calcule d'abord `(SELECT MIN(latitude) FROM Airports)`, qui vaut −90, puis l'utilise comme une valeur ordinaire dans le `WHERE`. On n'a plus besoin de recopier −90 à la main.
- La jointure avec `Countries` remplace la troisième requête (code `AQ` → nom du pays).

## 4 — (i) piste en asphalte la plus longue

#### 📝 ÉNONCÉ

> (i) Quelle est la ville où se trouve la piste en asphalte la plus longue ?

#### ✅ RÉSOLUTION

```sql
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Runways.type = 'ASP'
      AND Runways.length = (SELECT MAX(length) FROM Runways WHERE type = 'ASP');
```

*Résultat dans la console H2 :*

```text
CITY
Upington
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- ⚠️ La condition `Runways.type = 'ASP'` doit apparaître **dans la requête imbriquée** (pour calculer le maximum des pistes en asphalte, et non de toutes les pistes, qui vaut 30 000) **et dans la requête principale** (pour ne garder que la piste en asphalte qui a cette longueur).

## 4 — (j) en Europe

#### 📝 ÉNONCÉ

> (j) Quelle est la ville d'Europe où se trouve la piste en asphalte la plus longue ?

#### ✅ RÉSOLUTION

```sql
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP' AND Countries.continent = 'EU'
      AND Runways.length = (SELECT MAX(Runways.length) FROM Runways
                                JOIN Airports ON Airports.id = Runways.airport_id
                                JOIN Countries ON Countries.code = Airports.country
                                WHERE Runways.type = 'ASP' AND Countries.continent = 'EU');
```

*Résultat dans la console H2 :*

```text
CITY
Paris
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- C'est la combinaison des deux outils : les deux requêtes de la question 3 (j) emboîtées l'une dans l'autre.
- La requête imbriquée a besoin **des mêmes jointures** que la requête principale, puisque le continent est dans `Countries`.
- Conseil : écrire et tester d'abord la requête intérieure seule, puis l'emboîter.

---
# QUESTION 5

#### 📝 ÉNONCÉ

> 5. Télécharger et exécuter le fichier airports_tables_v2.sql et importer de nouveau les données du fichier airports_data.sql.

#### ✅ RÉSOLUTION

⏳ **Non traitée pour l'instant :** le lien vers `airports_tables_v2.sql` sur le site du professeur ne fonctionne pas encore (fichier absent du serveur). La méthode serait la même qu'à la question 1, avec `airports_tables_v2.sql` à la place de `airports_tables.sql`.

Les questions de la partie 6 n'utilisent que des attributs qui existent déjà dans les tables actuelles : elles sont résolues ci-dessous avec les données déjà chargées.

---
# QUESTION 6

> 6. Une autre série de questions...

## 6 (a)

#### 📝 ÉNONCÉ

> (a) Quel est le nombre de pistes de largeur supérieure ou égale à 100 m situées aux Seychelles ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Countries.name = 'Seychelles' AND Runways.width >= 100;
```

*Résultat dans la console H2 :*

```text
COUNT(*)
3
(1 row)
```

**Réponse :** 3 pistes (en prenant la valeur 100 comme le sujet).

> ⚠️ **Liberté prise :** dans le jeu de données, les largeurs sont en pieds et non en mètres ; on applique la valeur 100 de l'énoncé telle quelle (voir l'explication).

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT Airports.name, Runways.width FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    WHERE Airports.country = 'SC';
```

*Résultat dans la console H2 :*

```text
NAME                             | WIDTH
Alphonse Airport                 | 75
Assumption Island Airport        | 100
Desroches Airport                | 33
Farquhar Airport                 | 75
Seychelles International Airport | 151
Praslin Airport                  | 104
Coetivy Airport                  | 0
(7 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- Deux jointures pour relier une piste à son pays : `Runways` → `Airports` → `Countries`.
- `>=` signifie « supérieur ou égal » : la piste de largeur exactement 100 est comptée.
- **Remarque sur l'unité :** dans *OurAirports*, les largeurs sont **en pieds**. 100 m ≈ 328 pieds, et aucune piste des Seychelles n'est aussi large (la plus large fait 151 pieds ≈ 46 m). Avec `Runways.width >= 328`, la réponse serait 0. On suit ici l'énoncé, qui prend 100 comme valeur.

## 6 (b)

#### 📝 ÉNONCÉ

> (b) Quelles sont les villes françaises, classées par ordre alphabétique, qui ont un aéroport avec (au moins) une piste en herbe ?

#### ✅ RÉSOLUTION

```sql
SELECT DISTINCT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.country = 'FR' AND Runways.type = 'GRS'
    ORDER BY Airports.city;
```

*Résultat dans la console H2 :*

```text
CITY

Ambérieu
Arcachon/La Teste-de-Buch
Autun/Bellevue
Avaux
Avernes
Belpech
Corbeny
Coulommiers/Voisins
Fontrailles
Haguenau
Kogenheim
...
(38 rows)
```

**Réponse :** 38 lignes, d'*Ambérieu* à *Île d'Yeu* (la première ligne est vide : un aéroport dont la ville n'est pas renseignée).

#### 💡 EXPLICATION PÉDAGOGIQUE

- `ORDER BY attribut` **trie** le résultat (ordre croissant, donc alphabétique pour du texte). Il se place **à la fin** de la requête, après le `WHERE`.
- `DISTINCT` est indispensable : un aéroport peut avoir **plusieurs** pistes en herbe, et une ville plusieurs aéroports ; sans `DISTINCT`, la même ville apparaîtrait plusieurs fois.
- Herbe = `'GRS'` (code donné par le sujet, en majuscules dans les données).
- Les noms commençant par une lettre accentuée (*Écrosnes*, *Évaux*, *Île d'Yeu*) sont rangés **après le Z** : le tri compare les codes des caractères, et `É` a un code plus grand que `Z`.

## 6 (c)

#### 📝 ÉNONCÉ

> (c) Quels sont les noms des pays qui disposent de petits aéroports avec des pistes en gravier ?

#### ✅ RÉSOLUTION

Le sujet ne donne pas le code du gravier : on le cherche dans les données (`SELECT DISTINCT type FROM Runways;`). Le code le plus utilisé est `'GVL'` (*gravel*).

```sql
SELECT DISTINCT Countries.name FROM Countries
    JOIN Airports ON Countries.code = Airports.country
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.type = 'small_airport' AND Runways.type = 'GVL';
```

*Résultat dans la console H2 :*

```text
NAME
Australia
Belize
Brazil
Burkina Faso
Cambodia
Canada
Democratic Republic of the Congo
Eswatini
Finland
Greenland
Guinea
Haiti
...
(28 rows)
```

**Réponse :** 28 pays (Australia, Belize, Brazil, ..., United States).

> ⚠️ **Liberté prise :** le sujet ne donne pas le code du gravier ; on a choisi `'GVL'`, le code le plus utilisé dans les données (voir l'explication pour les autres écritures).

#### 💡 EXPLICATION PÉDAGOGIQUE

- « Petits aéroports » = `Airports.type = 'small_airport'` ; « pistes en gravier » = `Runways.type = 'GVL'`. Le préfixe est indispensable : les deux relations ont un attribut `type`.
- **Regard critique :** le gravier est aussi codé `'GRVL'`, `'GRAVEL'`, `'Gravel'`... selon les contributeurs. Pour les inclure, on pourrait écrire `Runways.type IN ('GVL', 'GRVL', 'GRAVEL')` (le mot-clé `IN` teste l'appartenance à une liste ; il est hors programme).

## 6 (d)

#### 📝 ÉNONCÉ

> (d) Quelles sont les villes européennes qui disposent de pistes en asphalte éclairées ?

#### ✅ RÉSOLUTION

```sql
SELECT DISTINCT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Countries.continent = 'EU' AND Runways.type = 'ASP' AND Runways.lighted = 1;
```

*Résultat dans la console H2 :*

```text
CITY

Aalborg
Aars
Abakan
Aberdeen
Aberporth
Agen/La Garenne
Agrinion
Ajaccio/Napoléon Bonaparte
Akureyri
Alajero, La Gomera Island
Albacete
...
(762 rows)
```

**Réponse :** 762 villes (Aalborg, Aberdeen, ...). Le nombre s'obtient en remplaçant `DISTINCT Airports.city` par `COUNT(DISTINCT Airports.city)`.

#### 💡 EXPLICATION PÉDAGOGIQUE

- Trois conditions reliées par `AND`, sur des attributs de trois relations différentes : c'est possible grâce aux deux jointures.
- « Éclairée » : d'après le sujet, `lighted` vaut **1** si la piste est éclairée (0 si non, −1 si inconnu). On écrit donc `lighted = 1`, et non `lighted = 'oui'` ou `lighted = True`.

## 6 (e)

#### 📝 ÉNONCÉ

> (e) Dans quelle ville de l'hémisphère sud se trouve l'aéroport comportant la plus longue piste ?

#### ✅ RÉSOLUTION

```sql
SELECT Airports.city FROM Airports
    JOIN Runways ON Airports.id = Runways.airport_id
    WHERE Airports.latitude < 0
      AND Runways.length = (SELECT MAX(Runways.length) FROM Runways
                                JOIN Airports ON Airports.id = Runways.airport_id
                                WHERE Airports.latitude < 0);
```

*Résultat dans la console H2 :*

```text
CITY
Upington
(1 row)
```

**Réponse :** *Upington* (Afrique du Sud).

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Hémisphère sud** = sous l'équateur = **latitude négative** : `Airports.latitude < 0`.
- Même structure que la question 4 (i) : une requête imbriquée calcule la longueur maximale parmi les pistes de l'hémisphère sud, puis on cherche la ville correspondante.
- On retrouve Upington : sa piste en asphalte est aussi la plus longue de l'hémisphère sud, tous types confondus.

## 6 (f)

#### 📝 ÉNONCÉ

> (f) Quels sont, pour chaque continent, les nombres cumulés de kilomètres de pistes en asphalte ?

#### ✅ RÉSOLUTION

```sql
SELECT Countries.continent, SUM(Runways.length) * 0.3048 / 1000 FROM Runways
    JOIN Airports ON Airports.id = Runways.airport_id
    JOIN Countries ON Countries.code = Airports.country
    WHERE Runways.type = 'ASP'
    GROUP BY Countries.continent;
```

*Résultat dans la console H2 :*

```text
CONTINENT | (SUM(RUNWAYS.LENGTH) * 0.3048) / 1000
AF        | 1751.052835200000000000000000
AS        | 2767.864720800000000000000000
EU        | 2460.018854400000000000000000
NA        | 7598.468623200000000000000000
OC        | 789.178101600000000000000000
SA        | 2800.257645600000000000000000
(6 rows)
```

**Réponse :** environ 1 751 km (AF), 2 768 km (AS), 2 460 km (EU), 7 598 km (NA), 789 km (OC), 2 800 km (SA). Aucune piste en asphalte n'est répertoriée en Antarctique avec ce code.

> ⚠️ **Liberté prise :** les longueurs étant en pieds, on les convertit en kilomètres (× 0,3048 ÷ 1000) ; le sujet ne précise pas l'unité.

#### 💡 EXPLICATION PÉDAGOGIQUE

- `SUM(...)` additionne les longueurs. Mais on veut **une somme par continent**, pas une seule somme pour le monde entier.
- **`GROUP BY` (hors programme)** regroupe les enregistrements qui ont la même valeur de `continent`, puis applique `SUM` **à chaque groupe** : on obtient une ligne par continent.
- **Conversion :** les longueurs sont en pieds ; × 0,3048 donne des mètres, puis ÷ 1000 des kilomètres. On peut faire des calculs directement dans le `SELECT`.
- Sans le `GROUP BY`, H2 refuse la requête : on ne peut pas afficher `continent` (une valeur par ligne) à côté de `SUM` (une seule valeur) sans dire comment regrouper.

## 6 (g)

#### 📝 ÉNONCÉ

> (g) Quelle est la région de France dénombrant le plus d'aéroports ?

#### ✅ RÉSOLUTION

```sql
SELECT Regions.name, COUNT(*) FROM Regions
    JOIN Airports ON Regions.code = Airports.region
    WHERE Regions.country = 'FR'
    GROUP BY Regions.name
    ORDER BY COUNT(*) DESC
    LIMIT 1;
```

*Résultat dans la console H2 :*

```text
NAME                 | COUNT(*)
Auvergne-Rhône-Alpes | 151
(1 row)
```

**Réponse :** *Auvergne-Rhône-Alpes*, avec 151 aéroports (tous types confondus, héliports et aéroports fermés compris).

*Vérification (non demandée), dans la console H2 :*

```sql
SELECT Regions.name, COUNT(*) FROM Regions
    JOIN Airports ON Regions.code = Airports.region
    WHERE Regions.country = 'FR'
    GROUP BY Regions.name
    ORDER BY COUNT(*) DESC;
```

*Résultat dans la console H2 :*

```text
NAME                       | COUNT(*)
Auvergne-Rhône-Alpes       | 151
Occitanie                  | 139
Nouvelle-Aquitaine         | 117
Grand Est                  | 102
Provence-Alpes-Côte-d'Azur | 84
Bourgogne-Franche-Comté    | 68
Hauts-de-France            | 67
Normandy                   | 56
Île-de-France              | 50
Centre-Val de Loire        | 45
Pays-de-la-Loire           | 43
Brittany                   | 40
...
(14 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- `GROUP BY Regions.name` + `COUNT(*)` : on compte les aéroports **de chaque région**.
- `ORDER BY COUNT(*) DESC` trie du plus grand au plus petit nombre (`DESC` = décroissant ; par défaut, l'ordre est croissant).
- `LIMIT 1` (hors programme) ne garde que la première ligne. Sans `LIMIT`, on lit simplement la première ligne du classement.
- La jointure se fait sur `Regions.code = Airports.region` : dans `Airports`, `region` contient le code de la région (`'FR-ARA'`), et non son nom.